# 27. 기존·신규 사례 통합 Leave-One-Product-Family-Out CV

기존 177건과 신규 89건을 하나의 데이터셋으로 통합하고, **통합 데이터 전체를 평가 대상으로 사용하는 교차검증**입니다.

- 원본 데이터: 기존 177건 + 신규 89건
- 중복 제거: 정규화한 Product Name 기준, 기존 데이터 우선 유지
- HS 분류 단위: HS6
- 검증 방식: Leave-One-Product-Family-Out
- 평가: 각 제품군을 한 번씩 통째로 제외하고 나머지 제품군으로 학습·검색
- 신규 사례도 Fold에 따라 학습 또는 평가 데이터에 포함

주의: Product Type 기반 제품군은 검색·검증용 휴리스틱입니다. HS 코드 자체를 이용해 제품군을 정의하지 않습니다. 신규 데이터의 제품군 매핑 오류나 제품 설명 결측은 성능에 영향을 줄 수 있습니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

In [ ]:
import re, json, zipfile, unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from kiwipiepy import Kiwi
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
BASE_FILE=quote('완제품_모델_기존177건_기준데이터.csv',safe='')
NEW_FILE=quote('CLIP_MTI227_신규89건_모델입력.csv',safe='')
KB_FILE='hs_knowledge_base.jsonl'
old_df=pd.read_csv(BASE+BASE_FILE,encoding='utf-8-sig').fillna('')
new_df=pd.read_csv(BASE+NEW_FILE,encoding='utf-8-sig').fillna('')
kb=pd.read_json(BASE+KB_FILE,lines=True).fillna('')
INPUT_COLUMNS=['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET='HS Code'
SUMMARY='Preprocessed Ruling Summary'
for data in [old_df,new_df]:
    for col in INPUT_COLUMNS+[TARGET,SUMMARY,'Product Type','Ruling Summary']:
        if col not in data: data[col]=''
        data[col]=data[col].astype(str).str.strip()
old_df=old_df[old_df[TARGET].ne('')].copy()
new_df=new_df[new_df[TARGET].ne('')].copy()
old_df['data_source']='existing_177'
new_df['data_source']='new_89'
all_df=pd.concat([old_df,new_df],ignore_index=True)
def normalize_name(x):
    x=unicodedata.normalize('NFKC',str(x)).lower()
    return re.sub(r'[^0-9a-z가-힣]+','',x)
all_df['name_key']=all_df['Product Name'].map(normalize_name)
all_df=all_df[all_df['name_key'].ne('')].copy()
all_df['duplicate_name']=all_df.duplicated('name_key',keep='first')
duplicate_audit=all_df[all_df['duplicate_name']].copy()
all_df=all_df[~all_df['duplicate_name']].copy().reset_index(drop=True)
all_df['hs_digits']=all_df[TARGET].map(lambda x:re.sub(r'\\D','',str(x)))
all_df['hs6']=all_df['hs_digits'].str[:6]
all_df['hs4']=all_df['hs_digits'].str[:4]
all_df=all_df[all_df['hs6'].str.len().eq(6)].copy().reset_index(drop=True)
all_df['duplicate_group']=pd.util.hash_pandas_object(all_df[INPUT_COLUMNS],index=False).astype(str)
print('기존:',len(old_df),'| 신규:',len(new_df),'| 이름 중복 제거:',len(duplicate_audit),'| 통합 고유 사례:',len(all_df))
display(duplicate_audit[['data_source','Product Name','HS Code']])


## 1. 제품군 정의

제품군은 기존 실험에서 사용한 Product Type 매핑 규칙을 유지합니다. 신규 사례도 동일한 규칙으로 매핑하여 Fold를 구성합니다.

In [ ]:
def assign_product_family(value):
    x=str(value).strip()
    if x in {'아이 메이크업 리무버','눈·입술 메이크업 제거'}: return 'eye_makeup_remover'
    if x in {'립 메이크업'}: return 'lip_makeup'
    if x in {'아이 메이크업'}: return 'eye_makeup'
    if x in {'화장품 함침 부직포'}: return 'impregnated_nonwoven_mask'
    if x in {'두발용 제품'}: return 'hair_care_styling'
    if x in {'영유아용 화장품'}: return 'baby_skincare'
    if x in {'방향·탈취 조제품'}: return 'indoor_air_freshener'
    if x in {'기초·기능성 화장품'}: return 'functional_skincare'
    if x in {'향수·화장수','네일용 제품','인체용 탈취제','목욕용 조제품','비누·세정용 제품','기타 화장용 제품','동물용 화장품'}: return 'basic_skincare'
    if '피내 주사용' in x or '필러' in x: return 'injectable_filler'
    if '향을 첨가한 종이' in x: return 'perfumed_paper'
    if '방향' in x: return 'vehicle_air_freshener' if '차량' in x else 'indoor_air_freshener'
    if '치과' in x or '불소 도포' in x or '산부식' in x: return 'dental_fluoride' if '불소' in x else 'dental_etchant'
    if '목욕' in x: return 'bath_foam' if '거품' in x else 'bath_oil_tablet'
    if '샴푸' in x: return 'shampoo_combo' if '겸용' in x else 'shampoo'
    if '염모' in x or '염색' in x or '컬러스프레이' in x: return 'hair_dye'
    if '헤어' in x or '두발' in x or '두피' in x: return 'hair_care_styling'
    if '립' in x or '입술' in x: return 'lip_balm' if '밤' in x else 'lip_exfoliant' if '스크럽' in x else 'lip_makeup'
    if '속눈썹' in x: return 'eyelash_care'
    if '마스카라' in x or '아이라이너' in x or '아이브로우' in x or '눈 화장' in x: return 'eye_makeup'
    if '볼연지' in x or '블러셔' in x: return 'blush'
    if '파우더' in x or '아이섀도' in x: return 'face_powder'
    if '부직포' in x or '물휴지' in x or '물티슈' in x or '마스크 팩' in x or '마스크팩' in x:
        return 'cosmetic_wipes' if '클렌징' in x or '물휴지' in x or '물티슈' in x else 'impregnated_nonwoven_mask'
    if '외음부' in x or '피어싱' in x or '이어홀' in x: return 'personal_cleansing'
    if '영유아' in x: return 'baby_skincare'
    if '클렌징' in x: return 'skincare_cleansing'
    if '선크림' in x or '자외선' in x: return 'suncare'
    if '팩' in x or '패치' in x or '마스크' in x: return 'skincare_mask_patch'
    if '기능성' in x: return 'functional_skincare'
    return 'basic_skincare'


all_df['product_family']=all_df['Product Type'].map(assign_product_family)
if all_df['product_family'].isna().any(): raise ValueError('제품군 매핑 누락')
all_df['query_text']=all_df[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
all_df['model_text']='[PRODUCT] '+all_df['Product Name']+' [INGREDIENTS_KR] '+all_df['Main Ingredients (KR)']+' [INGREDIENTS_EN] '+all_df['Main Ingredients (EN)']+' [FORM] '+all_df['Product Form and Packaging']+' [USE] '+all_df['Intended Use']
all_df['ruling_text']=(all_df['query_text']+' '+all_df[SUMMARY]).str.strip()
family_audit=all_df.groupby(['data_source','product_family']).agg(n=('hs6','size'),hs6_codes=('hs6',lambda x:', '.join(sorted(x.unique())))).reset_index()
display(family_audit)
print('통합 사례:',len(all_df),'| 제품군:',all_df['product_family'].nunique(),'| HS6:',all_df['hs6'].nunique())


## 2. 통합 데이터에서 평가 가능한 HS6 범위 확정

각 HS6 코드가 최소 4개 사례와 2개 이상의 제품군에 존재해야 Leave-One-Product-Family-Out 평가에서 학습·평가 양쪽에 나타날 수 있습니다. 조건을 충족하지 못한 코드는 별도 목록으로 저장하고 이번 성능평가에서는 제외합니다. 제외 기준은 결과를 확인하기 전에 고정합니다.

In [ ]:
class_audit=all_df.groupby('hs6').agg(n=('hs6','size'),product_families=('product_family','nunique'),unique_product_records=('duplicate_group','nunique')).reset_index()
class_audit['eligible']=(class_audit['n']>=4)&(class_audit['product_families']>=2)&(class_audit['unique_product_records']>=4)
display(class_audit.sort_values(['eligible','n'],ascending=[True,False]))
labels=sorted(class_audit.loc[class_audit['eligible'],'hs6'].tolist())
excluded_labels=class_audit[~class_audit['eligible']].copy()
eval_df=all_df[all_df['hs6'].isin(labels)].copy().reset_index(drop=True)
if not labels: raise ValueError('평가 가능한 HS6 클래스가 없습니다.')
label_family_counts=eval_df.groupby('hs6')['product_family'].nunique()
if (label_family_counts<2).any(): raise ValueError('단일 제품군에만 존재하는 HS6 코드가 포함되었습니다.')
print('평가 사례:',len(eval_df),'| HS6 클래스:',len(labels),'| 제품군:',eval_df['product_family'].nunique(),'| 제외 사례:',len(all_df)-len(eval_df))


## 3. 임베딩 및 지식베이스 준비

E5 사전학습 임베딩을 사용합니다. 각 Fold에서 검색 후보는 해당 Fold의 학습 제품군에 속한 사례로만 제한합니다. 따라서 평가 제품군의 상품 설명과 판정요지가 검색 후보에 들어가지 않습니다.

In [ ]:
kiwi=Kiwi()
STOPWORDS={'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
def remove_stopword_morphs(text):
    text=str(text); chars=list(text)
    for token in kiwi.tokenize(text):
        if token.form in STOPWORDS:
            for i in range(token.start,token.start+token.len): chars[i]=''
    return ''.join(chars)
for col in ['document_text','hs_code','rule_type','source_sheet']:
    if col not in kb.columns: kb[col]=''
    kb[col]=kb[col].fillna('').astype(str).str.strip()
kb['hs_digits']=kb['hs_code'].map(lambda x:re.sub(r'\\D','',str(x)))
kb_code=kb[kb['document_text'].ne('')&kb['hs_digits'].ne('')].reset_index(drop=True)
kb_code['search_text_sw']=(kb_code['document_text']+' '+kb_code['rule_type']+' '+kb_code['source_sheet']).map(remove_stopword_morphs)
kb_code['evidence_polarity']=kb_code['rule_type'].map(lambda x:'negative' if str(x).strip() in {'제외','제외·예외'} else 'mixed' if '제외' in str(x) and '포함' in str(x) else 'positive')
MODEL_NAME='intfloat/multilingual-e5-base'
sbert=SentenceTransformer(MODEL_NAME)
def encode_docs(texts):
    return sbert.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def encode_queries(texts):
    return sbert.encode(['query: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
query_emb=encode_queries(eval_df['query_text'].tolist())
product_emb=encode_docs(eval_df['query_text'].tolist())
ruling_emb=encode_docs(eval_df['ruling_text'].tolist())
kb_emb=encode_docs(kb_code['search_text_sw'].tolist())
print('평가 임베딩:',query_emb.shape,'| KB:',kb_emb.shape,'| device:',sbert.device)


## 4. HS6 후보 순위 함수

In [ ]:
def rank_case_codes(scores,train_codes):
    tmp=pd.DataFrame({'hs6':np.asarray(train_codes).astype(str),'score':np.asarray(scores)})
    return tmp.groupby('hs6')['score'].max().sort_values(ascending=False)
def code_compatible(doc_code,target_code):
    a=re.sub(r'\\D','',str(doc_code)); b=re.sub(r'\\D','',str(target_code))
    return bool(a and b and (a.startswith(b) or b.startswith(a)))
def rank_kb_codes(scores,method='max',top_k=5):
    values={}
    doc_codes=kb_code['hs_digits'].to_numpy()
    for label in labels:
        matched=np.asarray([float(scores[j]) for j,doc_code in enumerate(doc_codes) if code_compatible(doc_code,label)])
        if len(matched):
            matched=np.sort(matched)[::-1]
            values[label]=matched[0] if method=='max' else matched[:top_k].mean()
        else: values[label]=0.0
    return pd.Series(values,dtype=float).sort_values(ascending=False)
def weighted_rrf_rank(case_rank,kb_rank,k=60,kb_weight=0.75):
    case_pos={str(c):i+1 for i,c in enumerate(case_rank.index)}
    kb_pos={str(c):i+1 for i,c in enumerate(kb_rank.index)}
    values={c:1/(k+case_pos.get(c,100000))+kb_weight/(k+kb_pos.get(c,100000)) for c in labels}
    return pd.Series(values).sort_values(ascending=False)
def top_codes(rank,n=3): return [str(x) for x in rank.index[:n]]
def make_vectorizer():
    return FeatureUnion([
        ('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),
        ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))
    ])


## 5. 통합 데이터 Leave-One-Product-Family-Out CV

각 Fold에서 제품군 하나를 통째로 평가하고 나머지 제품군만 학습·검색에 사용합니다. 모든 평가 대상 HS6가 해당 Fold의 학습 데이터에도 존재하는지 확인합니다. 존재하지 않는 경우 해당 Fold의 점수를 임의로 산출하지 않고 오류로 중단합니다.

In [ ]:
logo=LeaveOneGroupOut()
splits=list(logo.split(eval_df,eval_df['hs6'],groups=eval_df['product_family']))
prediction_rows=[]
fold_rows=[]
for fold,(tr,te) in enumerate(splits,1):
    train,test=eval_df.iloc[tr].copy(),eval_df.iloc[te].copy()
    train_labels=set(train['hs6'])
    if not set(test['hs6']).issubset(train_labels): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS6 코드')
    fold_family=test['product_family'].iloc[0]
    fold_rows.append({'fold':fold,'heldout_family':fold_family,'train_n':len(tr),'test_n':len(te),'train_hs6_n':len(train_labels),'test_hs6_n':test['hs6'].nunique()})
    vectorizer=make_vectorizer()
    x_train=vectorizer.fit_transform(train['model_text'])
    x_test=vectorizer.transform(test['model_text'])
    svm=LinearSVC(C=1.0,class_weight='balanced').fit(x_train,train['hs6'])
    svm_scores=svm.decision_function(x_test)
    for local_i,global_i in enumerate(te):
        svm_rank=pd.Series(svm_scores[local_i],index=svm.classes_).sort_values(ascending=False)
        prediction_rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':fold_family,'data_source':eval_df.iloc[global_i]['data_source'],'true_hs6':eval_df.iloc[global_i]['hs6'],'true_hs4':eval_df.iloc[global_i]['hs4'],'model':'TFIDF_LinearSVC','predicted_hs6':str(svm_rank.index[0]),'top3_candidates':top_codes(svm_rank)})
    train_idx=np.asarray(tr)
    train_codes=train['hs6'].to_numpy()
    for global_i in te:
        q=query_emb[global_i]
        case_product=rank_case_codes(product_emb[train_idx]@q,train_codes)
        case_ruling=rank_case_codes(ruling_emb[train_idx]@q,train_codes)
        kb_scores=kb_emb@q
        kb_max=rank_kb_codes(kb_scores,'max')
        kb_top5=rank_kb_codes(kb_scores,'topk_mean',5)
        ranks={
            'SBERT_product_only':case_product,
            'SBERT_product_plus_ruling':case_ruling,
            'SBERT_KB_only':kb_max,
            'SBERT_product_plus_ruling_KBmax_w75':weighted_rrf_rank(case_ruling,kb_max,kb_weight=0.75),
            'SBERT_product_plus_ruling_KBtop5_w40':weighted_rrf_rank(case_ruling,kb_top5,kb_weight=0.40),
            'SBERT_product_plus_ruling_KBtop5_w50':weighted_rrf_rank(case_ruling,kb_top5,kb_weight=0.50),
            'SBERT_product_plus_ruling_KBtop5_w75':weighted_rrf_rank(case_ruling,kb_top5,kb_weight=0.75)
        }
        for model_name,rank in ranks.items():
            prediction_rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':fold_family,'data_source':eval_df.iloc[global_i]['data_source'],'true_hs6':eval_df.iloc[global_i]['hs6'],'true_hs4':eval_df.iloc[global_i]['hs4'],'model':model_name,'predicted_hs6':str(rank.index[0]) if len(rank) else '','top3_candidates':top_codes(rank)})
predictions=pd.DataFrame(prediction_rows)
fold_audit=pd.DataFrame(fold_rows)
display(fold_audit)
print('제품군 Fold:',len(splits),'| 평가 사례:',len(eval_df),'| 모델별 OOF 예측 수:',predictions.groupby('model').size().to_dict())


## 6. 성능 평가 및 데이터 출처별 진단

통합 OOF 전체 성능과 기존·신규 데이터 출처별 성능을 함께 확인합니다. 출처별 성능은 표본 구성 차이의 영향을 받으므로 단순한 우열로 해석하지 않습니다.

In [ ]:
def metric_summary(part):
    yt=part['true_hs6'].astype(str)
    yp=part['predicted_hs6'].astype(str)
    top3=np.mean([str(t) in [str(x) for x in c] for t,c in zip(yt,part['top3_candidates'])])
    return {'n':len(part),'hs4_accuracy':np.mean(yt.str[:4].to_numpy()==yp.str[:4].to_numpy()),'hs6_accuracy':accuracy_score(yt,yp),'macro_f1_hs6':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),'weighted_f1_hs6':f1_score(yt,yp,average='weighted',zero_division=0),'top3_hs6_accuracy':top3}
summary=pd.DataFrame([{'model':name,**metric_summary(part)} for name,part in predictions.groupby('model')]).sort_values('macro_f1_hs6',ascending=False)
display(summary.round(4))
source_metrics=predictions.groupby(['model','data_source']).apply(lambda x:pd.Series(metric_summary(x)),include_groups=False).reset_index()
display(source_metrics.sort_values(['model','data_source']))
family_metrics=predictions.groupby(['model','heldout_family']).apply(lambda x:pd.Series(metric_summary(x)),include_groups=False).reset_index()
display(family_metrics.sort_values(['model','heldout_family']))
class_metric_rows=[]
for model_name,part in predictions.groupby('model'):
    yt=part['true_hs6'].astype(str); yp=part['predicted_hs6'].astype(str)
    p,r,f,s=precision_recall_fscore_support(yt,yp,labels=labels,zero_division=0)
    for code,precision,recall,f1_value,support in zip(labels,p,r,f,s):
        class_metric_rows.append({'model':model_name,'hs6':code,'n':int(support),'precision':precision,'recall':recall,'f1':f1_value})
class_metrics=pd.DataFrame(class_metric_rows)
display(class_metrics.sort_values(['model','hs6']))


## 7. 혼동행렬 및 검색 근거 진단

In [ ]:
best_model=summary.iloc[0]['model']
best_predictions=predictions[predictions['model'].eq(best_model)].copy()
confusion=pd.crosstab(best_predictions['true_hs6'],best_predictions['predicted_hs6'],rownames=['true_hs6'],colnames=['predicted_hs6'])
display(confusion)
coverage_rows=[]
for label in labels:
    matched=kb_code[kb_code['hs_digits'].map(lambda x:code_compatible(x,label))]
    coverage_rows.append({'hs6':label,'compatible_kb_documents':len(matched),'positive_documents':int((matched['evidence_polarity']=='positive').sum()),'negative_documents':int((matched['evidence_polarity']=='negative').sum()),'mixed_documents':int((matched['evidence_polarity']=='mixed').sum())})
kb_coverage=pd.DataFrame(coverage_rows)
display(kb_coverage)
evidence_rows=[]
for row_idx,row in eval_df.iterrows():
    scores=kb_emb@query_emb[row_idx]
    for rank,j in enumerate(np.argsort(scores)[::-1][:10],1):
        doc=kb_code.iloc[j]
        evidence_rows.append({'row_index':int(row_idx),'data_source':row['data_source'],'product_family':row['product_family'],'product_name':row['Product Name'],'true_hs6':row['hs6'],'query_rank':rank,'similarity':float(scores[j]),'document_hs_code':doc['hs_code'],'rule_type':doc['rule_type'],'evidence_polarity':doc['evidence_polarity'],'source_sheet':doc['source_sheet'],'document_text':doc['document_text']})
kb_retrieval_evidence=pd.DataFrame(evidence_rows)
display(kb_retrieval_evidence.head(30))


## 8. 결과 저장

In [ ]:
OUT=Path('/content/hs_integrated_cv_27')
OUT.mkdir(parents=True,exist_ok=True)
predictions.to_csv(OUT/'27_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'27_model_comparison.csv',index=False,encoding='utf-8-sig')
source_metrics.to_csv(OUT/'27_source_metrics.csv',index=False,encoding='utf-8-sig')
family_metrics.to_csv(OUT/'27_family_metrics.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'27_class_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv(OUT/'27_fold_audit.csv',index=False,encoding='utf-8-sig')
family_audit.to_csv(OUT/'27_product_family_audit.csv',index=False,encoding='utf-8-sig')
class_audit.to_csv(OUT/'27_class_eligibility_audit.csv',index=False,encoding='utf-8-sig')
excluded_labels.to_csv(OUT/'27_excluded_classes.csv',index=False,encoding='utf-8-sig')
duplicate_audit.to_csv(OUT/'27_duplicate_name_audit.csv',index=False,encoding='utf-8-sig')
kb_coverage.to_csv(OUT/'27_kb_coverage.csv',index=False,encoding='utf-8-sig')
kb_retrieval_evidence.to_csv(OUT/'27_kb_retrieval_evidence.csv',index=False,encoding='utf-8-sig')
confusion.to_csv(OUT/'27_confusion_matrix_best_model.csv',encoding='utf-8-sig')
with zipfile.ZipFile('/content/27_integrated_cv_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob('*.csv'): z.write(f,arcname=f.name)
print('저장 완료: /content/27_integrated_cv_results.zip')


## 해석 시 주의사항

- 이번 실험은 기존 177건만 평가했던 25·26번과 평가 모집단 자체가 다릅니다. 따라서 Macro-F1을 직접적인 동일 조건 성능 향상으로 해석하지 않습니다.
- 중복 제거는 정규화 Product Name 기준이며, 이름이 다르지만 사실상 동일한 제품은 남아 있을 수 있습니다.
- 제품군은 Product Type 기반 규칙으로 정의했으므로 매핑 오류가 Fold 분리와 성능에 영향을 줄 수 있습니다.
- 평가 가능한 HS6는 사례 수 4건 이상, 제품군 2개 이상, 고유 제품 입력 4개 이상으로 사전에 제한합니다. 제외된 HS6는 별도 CSV로 확인합니다.
- 같은 제품군을 통째로 제외하므로 일반적인 무작위 K-fold보다 제품군 일반화에 초점을 둔 평가입니다.
- KB 가중치는 25·26번 탐색 결과를 참고해 0.40, 0.50, 0.75를 비교하지만, 이 통합 CV 결과에서 가장 좋은 값을 선택하면 선택 편향이 생깁니다. 최종 성능은 별도 홀드아웃 또는 추가 사례로 검증해야 합니다.